In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
# Load the CSV file into a DataFrame

human = pd.read_csv('human_rating.csv')
print(human.columns)

In [26]:
human_1 = human.drop([human.columns[0], human.columns[1], human.columns[-1]], axis=1)


In [ ]:
# Rename the columns with simpler names

human_1 = human_1.rename(columns={
    'Name (First Name)': 'name',
    'Please select Group ID:': 'group_id',
    'Please select the Scenario ID: ': 'scenario_id'
})
human_1 = human_1.rename(columns={
    human_1.columns[-3]: 'emotional_coherence',
    human_1.columns[-2]: 'emotional_adaptivity',
    human_1.columns[-1]: 'problem_resolution',
})

In [ ]:
# Rename for turn_rating columns

human_2 = pd.DataFrame()

# Create a new list of cleaned column names
new_columns = []

for col in human_1.columns:
    if re.match(r'^\d+\.\d+', col):
        prefix = re.match(r'^(\d+\.\d+)', col).group(1)
        new_columns.append(prefix)
    else:
        clean_name = col.split('\n')[0].split(' (')[0].strip()
        new_columns.append(clean_name)

# Assign the cleaned names back to human_1 (or to a copy)
human_2 = human_1.copy()
human_2.columns = new_columns

In [ ]:
# Verify the new column names
print(human_2.columns)

Index(['name', 'group_id', 'scenario_id', '1.1', '1.2', '1.3', '1.4', '1.5',
       '2.1', '2.2', '2.3', '2.4', '2.5', '3.1', '3.2', '3.3', '3.4', '3.5',
       '4.1', '4.2', '4.3', '4.4', '4.5', '5.1', '5.2', '5.3', '5.4', '5.5',
       'emotional_coherence', 'emotional_adaptivity', 'problem_resolution'],
      dtype='str')


In [ ]:
# Map scenario codes to condition names (scenario_id)
code_to_condition = {
    "F1": "L1_frustrated_multi",
    "N2": "L5_neutral_multi",
    "A3": "L10_angry_multi",
    "N1": "L1_neutral_multi",
    "A2": "L5_angry_multi",
    "N3": "L10_neutral_multi",
    "A1": "L1_angry_multi",
    "F2": "L5_frustrated_multi",
    "F3": "L10_frustrated_multi"
}
human_2['scenario_id'] = human_2['scenario_id'].map(code_to_condition)
human_2['scenario_id'].value_counts(dropna=False)

scenario_id
L1_frustrated_multi     3
L5_neutral_multi        3
L10_angry_multi         3
L10_neutral_multi       3
L5_frustrated_multi     3
L1_angry_multi          3
L10_frustrated_multi    3
L1_neutral_multi        2
L5_angry_multi          2
Name: count, dtype: int64

In [ ]:
# Check name column value counts to ensure everyone submitted the survey thrice
# Clean name just in case
human_2['name'] = (
    human_2['name']
    .str.strip()          # remove leading/trailing spaces
    .str.title()          # standardize capitalization: "anita" → "Anita"
    .fillna('Unknown')    # handle missing names
)
human_2['name'].value_counts(dropna=False)

name
Grace Le         3
Monn Thwe Han    3
Nick             3
Lauren           3
Helen            3
Anita            3
Cindy            3
Fnu              3
Jocelyn          1
Name: count, dtype: int64

In [ ]:
# Get unique raters (by name) so that it's not identity-based
unique_raters = human_2['name'].drop_duplicates().tolist()

# Create mapping: rater_name → rater_01, rater_02, ...
rater_to_id = {
    name: f"rater_{str(i+1).zfill(2)}"
    for i, name in enumerate(unique_raters)
}

# Apply to dataset
human_2['rater_id'] = human_2['name'].map(rater_to_id)

In [ ]:
# Reorder columns to have 'rater_id' as the second column

# Get current column list
cols = human_2.columns.tolist()

# Remove 'rater_id' from wherever it is
cols.remove('rater_id')

# Insert it at position 1 (second column, since indexing starts at 0)
cols.insert(1, 'rater_id')

# Reorder the DataFrame
human_2 = human_2[cols]
print(human_2.columns)

# Verify rater_id assignment
print(human_2['rater_id'].value_counts(dropna=False))

Index(['name', 'rater_id', 'group_id', 'scenario_id', '1.1', '1.2', '1.3',
       '1.4', '1.5', '2.1', '2.2', '2.3', '2.4', '2.5', '3.1', '3.2', '3.3',
       '3.4', '3.5', '4.1', '4.2', '4.3', '4.4', '4.5', '5.1', '5.2', '5.3',
       '5.4', '5.5', 'emotional_coherence', 'emotional_adaptivity',
       'problem_resolution'],
      dtype='str')


## Split the data into turning ratings and holistic ratings

In [101]:
metadata_human = human_2.iloc[:, 1:4].copy()
human_hol = human_2.iloc[:, -3:].copy()
human_turn = human_2.iloc[:, 1:-3].copy()

print(metadata_human.columns)
print(human_hol.columns)
print(human_turn.columns)

Index(['rater_id', 'group_id', 'scenario_id'], dtype='str')
Index(['emotional_coherence', 'emotional_adaptivity', 'problem_resolution'], dtype='str')
Index(['rater_id', 'group_id', 'scenario_id', '1.1', '1.2', '1.3', '1.4',
       '1.5', '2.1', '2.2', '2.3', '2.4', '2.5', '3.1', '3.2', '3.3', '3.4',
       '3.5', '4.1', '4.2', '4.3', '4.4', '4.5', '5.1', '5.2', '5.3', '5.4',
       '5.5'],
      dtype='str')


### Transform the turn data to a long format

In [ ]:
# Assign the turn and number_id columns (from the column names) by melting the DataFrame


# 1. Identify the rating columns (those matching 'X.Y')
rating_cols = [col for col in human_turn.columns if re.match(r'^\d+\.\d+$', str(col))]

# 2. Melt the DataFrame
human_turn_long = human_turn.melt(
    id_vars=['rater_id', 'group_id', 'scenario_id'],  # keep these as identifiers
    value_vars=rating_cols,                           # melt these columns
    var_name='item',                                  # name for the column that holds '1.1', '1.2', etc.
    value_name='score'                               # name for the actual score
)

# 3. Split 'item' into 'turn' and 'metric_id'
human_turn_long[['turn', 'number_id']] = human_turn_long['item'].str.split('.', expand=True).astype(int)

# 4. Drop the 'item' column (optional)
human_turn_long = human_turn_long.drop(columns=['item'])
print(human_turn_long.columns)

Index(['rater_id', 'group_id', 'scenario_id', 'score', 'turn', 'number_id'], dtype='str')


In [ ]:
# Map number_id to descriptive metric names

metric_labels = {
    1: 'tone_appropriateness',
    2: 'emotional_calibration',
    3: 'emotional_escalation',
    4: 'functional_empathy',
    5: 'contextual_appropriateness'
}
human_turn_long['metric_id'] = human_turn_long['number_id'].map(metric_labels)

# Reorder columns for consistent format
human_turn_long = human_turn_long[['rater_id', 'group_id', 'scenario_id', 'turn', 'metric_id','number_id', 'score']]

In [ ]:
# Clean the score column so that it contains only integers
human_turn_long['score'] = human_turn_long['score'].astype(str).str.extract(r'^(\d+)')[0].astype(int)

In [ ]:
# Check how many ratings we have per metric, per turn

print(human_turn_long['turn'].value_counts(dropna=False))
print(human_turn_long['metric_id'].value_counts(dropna=False))

turn
1    125
2    125
3    125
4    125
5    125
Name: count, dtype: int64
metric_id
tone_appropriateness          125
emotional_calibration         125
emotional_escalation          125
functional_empathy            125
contextual_appropriateness    125
Name: count, dtype: int64


In [ ]:
# Check the final DataFrame
human_turn_long.head()


,rater_id,group_id,scenario_id,turn,metric_id,number_id,score
0,rater_01,Group A,L1_frustrated_multi,1,tone_appropriateness,1,4
1,rater_01,Group A,L5_neutral_multi,1,tone_appropriateness,1,4
2,rater_01,Group A,L10_angry_multi,1,tone_appropriateness,1,4
3,rater_02,Group B,L10_neutral_multi,1,tone_appropriateness,1,3
4,rater_03,Group B,L10_neutral_multi,1,tone_appropriateness,1,4


### Transform the Holistic data to a long format

In [102]:
human_hol = pd.concat([metadata_human, human_hol], axis=1)
human_hol.head()

,rater_id,group_id,scenario_id,emotional_coherence,emotional_adaptivity,problem_resolution
0,rater_01,Group A,L1_frustrated_multi,3 (Adequate Memory): Generally maintains emot...,2 (Minimally Adaptive): Limited or delayed emo...,4 (Substantial Progress): Clear resolution pat...
1,rater_01,Group A,L5_neutral_multi,4 (Strong Memory): Maintains clear emotional c...,2 (Minimally Adaptive): Limited or delayed emo...,5 (Problem Resolved): Problem fully resolved o...
2,rater_01,Group A,L10_angry_multi,3 (Adequate Memory): Generally maintains emot...,2 (Minimally Adaptive): Limited or delayed emo...,2 (Minimal Progress): Some information provide...
3,rater_02,Group B,L10_neutral_multi,4 (Strong Memory): Maintains clear emotional c...,3 (Moderately Adaptive): Makes some appropriat...,4 (Substantial Progress): Clear resolution pat...
4,rater_03,Group B,L10_neutral_multi,4 (Strong Memory): Maintains clear emotional c...,4 (Responsive): Adjusts emotional tone approp...,5 (Problem Resolved): Problem fully resolved o...


In [ ]:
# Transform holistic ratings to long format to match turn-level ratings format

# 1. Identify holistic rating columns
global_cols = ['emotional_coherence', 'emotional_adaptivity', 'problem_resolution']

# 2. Melt the DataFrame
human_hol_long = human_hol.melt(
    id_vars=['rater_id', 'group_id', 'scenario_id'],
    value_vars=global_cols,
    var_name='metric_id',
    value_name='score'
)

# 3. Define mapping
metric_to_id = {
    'emotional_coherence': 6,
    'emotional_adaptivity': 7,
    'problem_resolution': 8
}

human_hol_long['number_id'] = human_hol_long['metric_id'].map(metric_to_id)
human_hol_long['turn'] = 0  # all holistic ratings have turn = 0


# 4. Reorder columns
human_hol_long = human_hol_long[['rater_id', 'group_id', 'scenario_id', 'turn', 'metric_id','number_id', 'score']]

# 5. Clean the score column
human_hol_long['score'] = human_hol_long['score'].astype(str).str.extract(r'^(\d+)')[0].astype(int)

In [ ]:
# Check how many ratings we have per metric
print(human_hol_long['metric_id'].value_counts(dropna=False))
print(human_hol_long['rater_id'].value_counts(dropna=False))


metric_id
emotional_coherence     25
emotional_adaptivity    25
problem_resolution      25
Name: count, dtype: int64
rater_id
rater_01    9
rater_02    9
rater_03    9
rater_04    9
rater_05    9
rater_06    9
rater_07    9
rater_08    9
rater_09    3
Name: count, dtype: int64


In [ ]:
# Check the final DataFrame
human_hol_long.head()

,rater_id,group_id,scenario_id,turn,metric_id,number_id,score
0,rater_01,Group A,L1_frustrated_multi,0,emotional_coherence,6,3
1,rater_01,Group A,L5_neutral_multi,0,emotional_coherence,6,4
2,rater_01,Group A,L10_angry_multi,0,emotional_coherence,6,3
3,rater_02,Group B,L10_neutral_multi,0,emotional_coherence,6,4
4,rater_03,Group B,L10_neutral_multi,0,emotional_coherence,6,4


In [100]:
total = human_hol_long['rater_id'].value_counts().sum()
total

np.int64(75)

## Save the clean data as CSV files

In [96]:
human_hol_long.to_csv('human_holistic.csv', index=False)
human_turn_long.to_csv('human_turn.csv', index=False)